# İP4 — Yapısal Gruplama Katsayısı (K_profil + K_etkinlik)

**Sorumlu:** Samet Koca | **Tarih:** 2 Temmuz 2026

**Girdi:** `data/processed/master_bloom.xlsx` (Sudenaz'ın İP3 çıktısı)

**Çıktı:** aynı dosyaya `K_profil` ve `K_etkinlik` sütunları eklenir

Bu notebook, dersin etkinlik profiline dayalı iki yapısal katsayı üretir. Bu katsayılar İP5'te kural tabanlı AKTS hesabında (`kural_akts`) kullanılacaktır.

### K_profil — Formül ve Mantık

$$K\_profil = 1 + 0.10 \times has\_lab + 0.15 \times has\_proje + 0.05 \times has\_uygulama$$

Neden bu ağırlıklar:
- **Proje (0.15):** En yüksek iş yükü — bağımsız çalışma, teslim edilebilir
- **Laboratuvar (0.10):** Orta yük — ekipman, ön hazırlık, rapor
- **Uygulama (0.05):** Düşük ek yük — ders içi pratik

Aralık: 1.00 (düz teorik) → 1.30 (üçü de dolu)

### K_etkinlik — Formül ve Mantık

$$K\_etkinlik = 1 + 0.03 \times (n\_etkinlik - ortalama\_n\_etkinlik)$$

Ortalamanın üstünde etkinliği olan ders 1'den büyük katsayı alır, altında olan 1'den küçük. Ortalama etkinlikli ders tam 1.00 alır.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Yol tanımları
BASE = Path.cwd().resolve().parents[1]  # course-credits-system köküne çık
IN_PATH = BASE / "data" / "processed" / "master_bloom.xlsx"
OUT_PATH = BASE / "data" / "processed" / "master_bloom.xlsx"

print("master_bloom.xlsx yükleniyor...")
df = pd.read_excel(IN_PATH, dtype={"Katalogid": str})
print(f"  {len(df)} satır, {len(df.columns)} sütun")

df.head(3)

master_bloom.xlsx yükleniyor...


  4133 satır, 107 sütun


,Katalogid,DersAdi,turid,derstur,Bolumid,BolumAd,EgitimYili,AKTSKredi,OgrenmeKazanim_1,OgrenmeKazanim_2,...,bloom_l5_ratio,bloom_l6_count,bloom_l6_ratio,bloom_avg_level,bloom_max_level,bloom_dominant_level,K_bilissel,bloom_justification,K_profil,K_etkinlik
0,201206,Veri Yapıları ve Algoritmaları,1,Teorik,201,Bilgisayar Mühendisliği,20252026,5,Birden fazla veri yapısını beraber kullanmayı ...,Temel veri yapıları ve algoritmalarını karşıla...,...,0.0,2,0.4000,3.6000,6,1,1.2600,Dominant: L1 | Ort: 3.60 | Maks: L6 | K=1.2600,1.00,0.9887
1,9907120,Temel Bilgisayar Bilimleri (Sosyal) - UE,21,e_ders,111,İngiliz Dili ve Edebiyatı,20252026,4,Mesleğinin gerektirdiği programları kullanır.,Sorun çözme adımlarını tespit eder.,...,0.0,0,0.0000,2.3333,3,3,1.1333,Dominant: L3 | Ort: 2.33 | Maks: L3 | K=1.1333,1.20,1.1087
2,9907119,Temel Bilgisayar Bilimleri (UE),21,e_ders,117,Biyoloji,20252026,4,Basit seviyede program yazar.,Program üzerindeki hataları tespit eder.,...,0.0,1,0.1429,2.2857,6,1,1.1286,Dominant: L1 | Ort: 2.29 | Maks: L6 | K=1.1286,1.15,1.1087


In [2]:
# ─── K_profil ───────────────────────────────────
# Ders profilinin yapısal yoğunluğu
# Lab, proje ve uygulama varlığına göre ağırlıklandırma
# Proje en yüksek ağırlık (0.15) çünkü en fazla iş yükü
df["K_profil"] = (
    1
    + 0.10 * df["has_lab"]
    + 0.15 * df["has_proje"]
    + 0.05 * df["has_uygulama"]
).round(4)

# ─── K_etkinlik ─────────────────────────────────
# Etkinlik yoğunluğunun ortalamadan sapması
ort_etkinlik = df["n_etkinlik"].mean()
df["K_etkinlik"] = (
    1 + 0.03 * (df["n_etkinlik"] - ort_etkinlik)
).round(4)

print(f"Ortalama n_etkinlik: {ort_etkinlik:.2f}")

# ─── Kaydet ─────────────────────────────────────
df.to_excel(OUT_PATH, index=False)
df.to_csv(OUT_PATH.with_suffix(".csv"), index=False, encoding="utf-8-sig")
print(f"Kaydedildi: {OUT_PATH}")
print(f"Toplam sütun: {len(df.columns)}")

df[["Katalogid", "DersAdi", "has_lab", "has_proje", "has_uygulama", "K_profil", "n_etkinlik", "K_etkinlik"]].head(3)

Ortalama n_etkinlik: 4.38


Kaydedildi: C:\Users\skoca\Workspace\projects\course-credits-system\data\processed\master_bloom.xlsx
Toplam sütun: 107


,Katalogid,DersAdi,has_lab,has_proje,has_uygulama,K_profil,n_etkinlik,K_etkinlik
0,201206,Veri Yapıları ve Algoritmaları,0,0,0,1.00,4,0.9887
1,9907120,Temel Bilgisayar Bilimleri (Sosyal) - UE,0,1,1,1.20,8,1.1087
2,9907119,Temel Bilgisayar Bilimleri (UE),1,0,1,1.15,8,1.1087


### Sonuçlar

K_profil: Min 1.00, Max 1.30, Ort 1.019, yüksek yoğunluk (>1.20): 25 ders

K_etkinlik: Min 0.87, Max 1.17, Ort 1.00

In [3]:
print("=== K_profil İstatistikleri ===")
print(f"  Min : {df['K_profil'].min():.4f}")
print(f"  Max : {df['K_profil'].max():.4f}")
print(f"  Ort : {df['K_profil'].mean():.4f}")
print(f"  K_profil > 1.20 (yüksek yoğunluk): {(df['K_profil'] > 1.20).sum()} ders")

print("\n=== K_etkinlik İstatistikleri ===")
print(f"  Min : {df['K_etkinlik'].min():.4f}")
print(f"  Max : {df['K_etkinlik'].max():.4f}")
print(f"  Ort : {df['K_etkinlik'].mean():.4f}")

print("\n=== Fakülte bazlı ortalama K_profil ===")
fakulte_ort = df.groupby("Fakulte")["K_profil"].mean().round(4).sort_values(ascending=False)
display(fakulte_ort.to_frame("K_profil_ortalama"))

print("\n=== K_profil Dağılımı (value_counts) ===")
print(df["K_profil"].value_counts().sort_index().to_string())

=== K_profil İstatistikleri ===
  Min : 1.0000
  Max : 1.3000
  Ort : 1.0190
  K_profil > 1.20 (yüksek yoğunluk): 25 ders

=== K_etkinlik İstatistikleri ===
  Min : 0.8687
  Max : 1.1687
  Ort : 1.0000

=== Fakülte bazlı ortalama K_profil ===


,K_profil_ortalama
Fakulte,
Teknoloji,1.0378
Mühendislik,1.0317
Spor Bilimleri,1.0181
Güzel Sanatlar,1.0171
Eğitim,1.0123
Fen-Edebiyat,1.0076
Siyasal Bilgiler,1.0015



=== K_profil Dağılımı (value_counts) ===
K_profil
1.00    3384
1.05     332
1.10     103
1.15     256
1.20      33
1.25      19
1.30       6


### Bulgu ve İP5'e Devir

Teknoloji ve Mühendislik en yüksek K_profil'e sahip — lab/proje yoğun oldukları için. Bu H1 hipotezini destekler.

İP5'te

$$kural\_akts = akts\_mevcut \times K\_etkinlik \times K\_profil \times K\_bilissel$$

formülünde bu katsayılar kullanılacak.